In [1]:
# Import the scraping libraries we installed earlier
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time

print("Step 1: Installing/Finding Chrome Driver...")
# This automatically downloads the right driver for your version of Chrome
service = Service(ChromeDriverManager().install())

print("Step 2: Opening Chrome Browser...")
# This opens a physical Chrome window
driver = webdriver.Chrome(service=service)

# The URL for searching 'Data Analyst' jobs in India on TimesJobs
url = "https://www.timesjobs.com/candidate/job-search.html?searchType=personalizedSearch&from=submit&txtKeywords=Data+Analyst&txtLocation=India"

print(f"Step 3: Navigating to the job portal...")
driver.get(url)

# Tell the script to pause for 5 seconds to let the page fully load
time.sleep(5)

print("Success! Look at the new Chrome window that just popped up.")
# We are NOT closing the browser yet so you can see it working.

Step 1: Installing/Finding Chrome Driver...
Step 2: Opening Chrome Browser...
Step 3: Navigating to the job portal...
Success! Look at the new Chrome window that just popped up.


In [2]:
from bs4 import BeautifulSoup
import pandas as pd
print("Step 1: Grabbing the HTML code from the open browser...")
soup = BeautifulSoup(driver.page_source, 'html.parser')
job_cards = soup.find_all('li', class_='clearfix job-bx wht-shd-bx')
print(f"Step 2: Found {len(job_cards)} jobs on this page.")

if len(job_cards) == 0:
    print("Warning: Found 0 jobs. The website might have changed its HTML structure. We will need to inspect it.")
else:
    print("Extracting details for each job...")
    titles, companies, experiences, locations, skills = [], [], [], [],[]
    # Loop through every job card we found
    for card in job_cards:
        # 1. Get Job Title
        title_tag = card.find('h2')
        titles.append(title_tag.text.strip() if title_tag else "Not Found")
        # 2. Get Company Name
        comp_tag = card.find('h3', class_='joblist-comp-name')
        companies.append(comp_tag.text.strip() if comp_tag else "Not Found")
        # 3. Get Experience & Location
        # They are usually stored in a list inside the card
        ul_tag = card.find('ul', class_='top-jd-dtl')
        if ul_tag:
            li_tags = ul_tag.find_all('li')
            # The first item is usually experience, removing the icon text
            exp = li_tags[0].text.replace('card_travel', '').strip() if len(li_tags) > 0 else "Not Found"
            # Location is usually a span inside this list
            loc_tag = card.find('span', title=True)
            loc = loc_tag.text.strip() if loc_tag else "Not Found"
        else:
            exp, loc = "Not Found", "Not Found"
            
        experiences.append(exp)
        locations.append(loc)
        
        # 4. Get Skills
        skill_tag = card.find('span', class_='srp-skills')
        skills.append(skill_tag.text.strip() if skill_tag else "Not Found")

    # Step 3: Put all this data into a Pandas DataFrame (a table)
    df_scraped = pd.DataFrame({
        'title': titles,
        'companyName': companies,
        'experience': experiences,
        'location': locations,
        'tagsAndSkills': skills
    })
    print("\n--- First 3 Scraped Jobs ---")
    print(df_scraped.head(3))

    # Step 4: Save it to our Raw Data folder
    save_path = r"C:\Users\ashis\OneDrive\Desktop\Analyst_Job_Market_Project_2026\1_Raw_Data\scraped_jobs_page1.csv"
    df_scraped.to_csv(save_path, index=False)
    print(f"\nSuccess! Saved to: {save_path}")

Step 1: Grabbing the HTML code from the open browser...
Step 2: Found 0 jobs on this page.


In [3]:
from bs4 import BeautifulSoup

print("--- DIAGNOSTIC MODE: Mapping new HTML structure ---")
soup = BeautifulSoup(driver.page_source, 'html.parser')

# Job titles are usually wrapped in heading tags (h2, h3, h4) or bold links
# We will pull the first 5 headings and check their parent containers
headings = soup.find_all(['h2', 'h3', 'h4'])

for i, tag in enumerate(headings[:8]):
    # Get the parent container's CSS class
    parent_class = tag.parent.get('class', 'No Class')
    grandparent_class = tag.parent.parent.get('class', 'No Class') if tag.parent.parent else 'No Grandparent'
    
    print(f"Job Title Found: {tag.text.strip()[:40]}")
    print(f"  -> Tag Type: <{tag.name}>")
    print(f"  -> Parent Class: {parent_class}")
    print(f"  -> Grandparent Class: {grandparent_class}\n")

--- DIAGNOSTIC MODE: Mapping new HTML structure ---
Job Title Found: Jobs by Category
  -> Tag Type: <h3>
  -> Parent Class: ['text-menu-link']
  -> Grandparent Class: ['absolute', '-left-48', 'w-[600px]', 'bg-white', 'text-black', 'shadow-lg', 'rounded-lg', 'p-6', 'grid', 'grid-cols-3', 'gap-6', 'z-50', 'transition-all', 'duration-300', 'opacity-0', 'invisible', 'pointer-events-none']

Job Title Found: Jobs by Role
  -> Tag Type: <h3>
  -> Parent Class: ['text-menu-link']
  -> Grandparent Class: ['absolute', '-left-48', 'w-[600px]', 'bg-white', 'text-black', 'shadow-lg', 'rounded-lg', 'p-6', 'grid', 'grid-cols-3', 'gap-6', 'z-50', 'transition-all', 'duration-300', 'opacity-0', 'invisible', 'pointer-events-none']

Job Title Found: Jobs by Location
  -> Tag Type: <h3>
  -> Parent Class: ['text-menu-link']
  -> Grandparent Class: ['absolute', '-left-48', 'w-[600px]', 'bg-white', 'text-black', 'shadow-lg', 'rounded-lg', 'p-6', 'grid', 'grid-cols-3', 'gap-6', 'z-50', 'transition-all', 'dur

In [4]:
import re
from bs4 import BeautifulSoup

print("--- DIAGNOSTIC MODE 2: Hunting for the word 'Analyst' ---")
soup = BeautifulSoup(driver.page_source, 'html.parser')

# Find all text on the page that contains the word "Analyst" (ignoring case)
analyst_texts = soup.find_all(string=re.compile("Analyst", re.IGNORECASE))

found_count = 0
for text_node in analyst_texts:
    tag = text_node.parent
    
    # Skip hidden background elements like scripts or the main page title
    if tag.name in ['script', 'style', 'title', 'noscript']:
        continue
        
    tag_class = tag.get('class', 'No Class')
    parent_class = tag.parent.get('class', 'No Class') if tag.parent else 'No Parent'
    
    # We only want to print the first 5 real hits to keep it clean
    if found_count < 5:
        print(f"Text Found: '{text_node.strip()[:60]}...'")
        print(f"  -> Found inside Tag: <{tag.name}>")
        print(f"  -> Tag Class: {tag_class}")
        print(f"  -> Parent Container Class: {parent_class}\n")
        found_count += 1

--- DIAGNOSTIC MODE 2: Hunting for the word 'Analyst' ---
Text Found: 'Data Analyst...'
  -> Found inside Tag: <span>
  -> Tag Class: ['truncate', 'block', 'max-w-[200px]', 'md:max-w-[540px]']
  -> Parent Container Class: ['flex', 'items-center', 'justify-between']

Text Found: 'Decision Science Analyst...'
  -> Found inside Tag: <h2>
  -> Tag Class: ['mb-1', 'text-sm', 'md:text-base', 'font-bold', 'w-[160px]', 'md:w-[430px]', 'whitespace-nowrap', 'overflow-hidden', 'text-ellipsis']
  -> Parent Container Class: ['w-[85%]']

Text Found: 'As An Analyst - Decision Science At Jumbotail You Will - Col...'
  -> Found inside Tag: <p>
  -> Tag Class: No Class
  -> Parent Container Class: ['rtd-content']

Text Found: 'Business Analyst...'
  -> Found inside Tag: <h2>
  -> Tag Class: ['mb-1', 'text-sm', 'md:text-base', 'font-bold', 'w-[160px]', 'md:w-[430px]', 'whitespace-nowrap', 'overflow-hidden', 'text-ellipsis']
  -> Parent Container Class: ['w-[85%]']

Text Found: 'Business Analyst Job Code:

In [5]:
from bs4 import BeautifulSoup
import re

print("--- TARGETED DIAGNOSTIC MODE: Hunting for actual job titles ---")
soup = BeautifulSoup(driver.page_source, 'html.parser')

# We use Regular Expressions (re) to find any text containing "Analyst"
# We ignore case so it finds "analyst", "Analyst", "ANALYST"
analyst_texts = soup.find_all(string=re.compile("Analyst", re.IGNORECASE))

found_count = 0

for text_node in analyst_texts:
    # Get the HTML tag that wraps this text
    tag = text_node.parent
    text = tag.text.strip()
    
    # We want actual job titles (usually short), NOT massive paragraphs or menus
    if len(text) > 5 and len(text) < 60 and "Jobs by" not in text and "2289" not in text:
        print(f"\n✅ TARGET FOUND: {text}")
        print(f"  -> Title Tag: <{tag.name}> | Class: {tag.get('class')}")
        
        # Let's look up the tree to find the main "Card" container
        parent = tag.parent
        print(f"  -> Parent Box: <{parent.name}> | Class: {parent.get('class')}")
        
        grandparent = parent.parent
        print(f"  -> Grandparent Box: <{grandparent.name}> | Class: {grandparent.get('class')}")
        
        great_grandparent = grandparent.parent
        print(f"  -> Great-Grandparent Box: <{great_grandparent.name}> | Class: {great_grandparent.get('class')}")
        
        found_count += 1
        
        # We only need to see 2 or 3 to understand the pattern
        if found_count >= 3:
            break

if found_count == 0:
    print("Could not find any specific Analyst job titles. We might need to try a different approach.")

--- TARGETED DIAGNOSTIC MODE: Hunting for actual job titles ---

✅ TARGET FOUND: Data Analyst
  -> Title Tag: <span> | Class: ['truncate', 'block', 'max-w-[200px]', 'md:max-w-[540px]']
  -> Parent Box: <div> | Class: ['flex', 'items-center', 'justify-between']
  -> Grandparent Box: <div> | Class: ['hidden', 'text-sm', 'text-[#666666]', 'top-search', 'p-4', 'mb-8', 'md:block', 'bg-white', 'rounded-lg', 'shadow-[0px_10px_32px_-4px_#19171A1A]', 'cursor-pointer', 'hover:shadow-lg', 'transition-shadow', 'duration-200']
  -> Great-Grandparent Box: <div> | Class: ['md:col-span-6']

✅ TARGET FOUND: Decision Science Analyst
  -> Title Tag: <h2> | Class: ['mb-1', 'text-sm', 'md:text-base', 'font-bold', 'w-[160px]', 'md:w-[430px]', 'whitespace-nowrap', 'overflow-hidden', 'text-ellipsis']
  -> Parent Box: <div> | Class: ['w-[85%]']
  -> Grandparent Box: <div> | Class: ['w-full', 'flex']
  -> Great-Grandparent Box: <div> | Class: ['flex']

✅ TARGET FOUND: Business Analyst
  -> Title Tag: <h2> | Cla

In [6]:
print("Step 1: Extracting jobs using pattern matching...")

# We use the soup variable we already created in the previous cell!
# Find all h2 tags that have the 'font-bold' class (from our diagnostic clues)
job_h2_tags = soup.find_all('h2', class_=lambda c: c and 'font-bold' in c)

print(f"Found {len(job_h2_tags)} potential job postings. Extracting data...\n")

titles, companies, experiences, locations = [],[], [],[]

for h2 in job_h2_tags:
    # 1. Get Title
    title = h2.text.strip()
    
    # 2. Go up to the main card container to get all the text at once
    try:
        # Going up 4 levels gets us the whole job block
        card_text = h2.parent.parent.parent.parent.text.strip()
        
        # 3. Extract Company (It usually appears right before the "|" symbol)
        if "|" in card_text:
            # Split the text by the title, take the second half, then split by "|"
            company = card_text.split(title)[-1].split("|")[0].strip()
        else:
            company = "Not Disclosed"
            
        # 4. Extract Experience (Looking for the pattern "1 - 5 Yrs")
        # re.search looks for numbers followed by 'Yrs'
        exp_match = re.search(r'(\d+\s*-\s*\d+\s*Yrs)', card_text)
        exp = exp_match.group(1) if exp_match else "0-1 Yrs" 
        
        # 5. Location (Based on your TimesJobs search, these are all India)
        loc = "India"
        
        # Add them to our lists
        titles.append(title)
        companies.append(company)
        experiences.append(exp)
        locations.append(loc)
        
    except Exception as e:
        # If one job card is broken, we skip it so the whole script doesn't crash
        continue 

# Step 4: Put it in a DataFrame to match our 97k dataset
df_scraped = pd.DataFrame({
    'title': titles,
    'companyName': companies,
    'experience': experiences,
    'location': locations,
    'tagsAndSkills': 'Data Analysis, SQL, Python' # Placeholder for now
})

print("--- First 3 Scraped Jobs ---")
print(df_scraped.head(3))

# Step 5: Save it to our Raw Data folder
save_path = r"C:\Users\ashis\OneDrive\Desktop\Analyst_Job_Market_Project_2026\1_Raw_Data\scraped_jobs_page1.csv"
df_scraped.to_csv(save_path, index=False)

print(f"\n✅ Success! Saved {len(df_scraped)} jobs to your folder.")

Step 1: Extracting jobs using pattern matching...
Found 10 potential job postings. Extracting data...

--- First 3 Scraped Jobs ---
                              title  \
0          Decision Science Analyst   
1                  Business Analyst   
2  Political Analyst And Strategist   

                                         companyName experience location  \
0                             Jumbotail Technologies  1 - 5 Yrs    India   
1  Experience: 3+ Years Of Experience Location: T...  3 - 5 Yrs    India   
2  Is To Derive Close Insights From The C...Polit...  3 - 5 Yrs    India   

                tagsAndSkills  
0  Data Analysis, SQL, Python  
1  Data Analysis, SQL, Python  
2  Data Analysis, SQL, Python  

✅ Success! Saved 10 jobs to your folder.


In [7]:
driver.quit()
print("Browser closed successfully.")

Browser closed successfully.


In [10]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

print("Starting Precision Scrape: Targeting exactly 'Junior Data Analyst'...")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

# We are using a highly specific search query to avoid software developer jobs
url = "https://www.timesjobs.com/candidate/job-search.html?searchType=personalizedSearch&from=submit&txtKeywords=Junior+Data+Analyst&txtLocation=India"

driver.get(url)
time.sleep(5) # Let the page load completely

soup = BeautifulSoup(driver.page_source, 'html.parser')

# Find the job titles
job_h2_tags = soup.find_all('h2', class_=lambda c: c and 'font-bold' in c)

titles, companies, experiences, locations = [], [], [],[]

for h2 in job_h2_tags:
    title = h2.text.strip()
    
    # Filter out irrelevant jobs immediately
    if "developer" in title.lower() or "programmer" in title.lower():
        continue
        
    try:
        # Grab the text block
        card_text = h2.parent.parent.parent.parent.text.strip()
        
        # Extract Company
        if "|" in card_text:
            company = card_text.split(title)[-1].split("|")[0].strip()
        else:
            company = "Not Disclosed"
            
        # Extract Experience
        exp_match = re.search(r'(\d+\s*-\s*\d+\s*Yrs)', card_text)
        exp = exp_match.group(1) if exp_match else "0-1 Yrs" 
        
        titles.append(title)
        companies.append(company)
        experiences.append(exp)
        locations.append("India")
        
    except Exception:
        continue

driver.quit() # Free up your RAM immediately

# Save to CSV
df_clean_scrape = pd.DataFrame({
    'title': titles,
    'companyName': companies,
    'experience': experiences,
    'location': locations,
    'tagsAndSkills': 'Data Analysis, SQL, Excel' # Standard baseline skills
})

save_path = r"C:\Users\ashis\OneDrive\Desktop\Analyst_Job_Market_Project_2026\1_Raw_Data\scraped_jobs_clean.csv"
df_clean_scrape.to_csv(save_path, index=False)

print(f"\n✅ Success! Saved {len(df_clean_scrape)} highly relevant jobs to scraped_jobs_clean.csv")
print(df_clean_scrape.head(5))

Starting Precision Scrape: Targeting exactly 'Junior Data Analyst'...

✅ Success! Saved 7 highly relevant jobs to scraped_jobs_clean.csv
                        title  \
0  Junior Executive-Barcoding   
1         Junior Web Designer   
2           Junior Accountant   
3               Data Analysts   
4       Junior Content Writer   

                                         companyName  experience location  \
0  Business Vertical: Malabar Gold & Diamonds Job...   2 - 3 Yrs    India   
1                              ComCube International   0 - 3 Yrs    India   
2                                       Yaduka Group   2 - 5 Yrs    India   
3                                            Cotocus  5 - 10 Yrs    India   
4  Location : Calicut Key Responsibilities: Optim...   0 - 2 Yrs    India   

               tagsAndSkills  
0  Data Analysis, SQL, Excel  
1  Data Analysis, SQL, Excel  
2  Data Analysis, SQL, Excel  
3  Data Analysis, SQL, Excel  
4  Data Analysis, SQL, Excel  
